# Import document

In [1]:
import json
import tiktoken

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader,TextLoader,PyPDFDirectoryLoader
from langchain_community.vectorstores import Chroma
from dotenv.ipython import load_dotenv
import os
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_mistralai import ChatMistralAI
from langchain_huggingface import HuggingFaceEmbeddings


/home/diabate/Bureau/supply-chain-contract-rag-agent/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_696567/389804495.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader,TextLoader,PyPDFDirectoryLoader


# Importer PDF de souvenir entre moi et ma moitié

In [2]:
path_to_pdf = "./remenber/"

loader = PyPDFDirectoryLoader(path_to_pdf, glob="**/*.pdf")

# Splitters

In [3]:
splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=1000,
    chunk_overlap=200,
    encoding_name="cl100k_base"
)

# Spliter document or Tokenize

In [4]:
docs = loader.load_and_split(splitter)

# Importer embedding model

In [5]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2932.56it/s]


# Apply Embedding of docs

In [6]:
vectorstore = Chroma.from_documents(
    docs, 
    embeddings,
    collection_name="remenber_collection",
    persist_directory="./vectorstore_remenber"
    )

# Retriever

In [7]:
retriever = vectorstore.as_retriever(
    type="similarity",
    search_kwargs={"k": 5})

# Define fonction RAG

In [86]:
prompt_template = """Tu es un assistant d'analyse de relation strictement neutre, objectif et formel. 
Ton rôle est d'analyser 4 ans d'historique de chat WhatsApp entre **Diabaté** et **Thera**.

RÈGLES STRICTES DE NEUTRALITÉ ET D'IDENTITÉ :
1. **Pas de pronoms directs** : N'utilise JAMAIS de pronoms comme "Vous", "Tu", "Je" ou "Moi" pour t'adresser à l'utilisateur. 
2. **Troisième personne uniquement** : Réfère-toi toujours aux deux individus par leurs prénoms respectifs : **Diabaté** et **Thera**. Le système doit ignorer qui est en train de poser la question (l'application étant accessible aux deux).
3. **Langue obligatoire** : Tu dois répondre EXCLUSIVEMENT en français. L'anglais est strictement interdit.

RÈGLES DE COMPRÉHENSION (HUMOUR VS SÉRIEUX) :
4. **Détection du second degré** : Les discussions contiennent des taquineries et des blagues. Ne prends pas les provocations au premier degré (par exemple, les piques sur la santé ou les petites bouderies).
5. **Poids des déclarations** : Accorde plus d'importance aux longs messages sérieux (comme les anniversaires) qu'aux réponses courtes du quotidien.

RÈGLES D'AFFICHAGE ET DE CONCISION :
- **Conclusion directe d'abord** : Commence immédiatement par 1 ou 2 phrases claires qui résument la réponse à la question. Pas d'introduction inutile.
- **Le bloc de dialogue fluide** : Affiche l'extrait textuel sous la forme d'un script continu (entre 3 et 7 répliques maximum) pour que la discussion reste drôle et fluide à relire.
- **Format des messages** : 
  > **[JJ/MM/AAAA - HH:MM] Nom** : "Message"
- **Analyse finale courte** : Termine par un résumé en 2 ou 3 points clés maximum pour expliquer le contexte de l'échange, sans sur-analyser chaque mot.

<historique_recent>
{chat_history}
</historique_recent>

<contexte_documents>
{context}
</contexte_documents>

Question : {question}

Réponse (Directe, formelle, structurée, rédigée uniquement à la troisième personne et 100% en français) :
"""

In [60]:
from langchain_mistralai import ChatMistralAI
key = os.getenv("CUAD_KEY")


llm = ChatMistralAI(
    model="mistral-small-latest", 
    temperature=0.0,
    api_key=key
)

In [87]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.messages import HumanMessage, AIMessage

# 1. Initialisation de la mémoire en dehors de la fonction (pour qu'elle persiste)
if 'memory_history' not in locals():
    memory_history = InMemoryChatMessageHistory()

def agent_remember(user_question, llm, prompt_template):
    # a. Récupération des documents pertinents (RAG)
    relevant_docs = retriever.invoke(user_question)
    context = "\n\n".join([doc.page_content for doc in relevant_docs])
    context = context.replace(" [", "\n[")
    
    # b. Extraction et limitation de l'historique aux 10 derniers échanges (20 messages)
    all_messages = memory_history.messages
    last_20_messages = all_messages[-20:] if len(all_messages) > 20 else all_messages
    
    # Formatage visuel de l'historique pour le prompt
    chat_history_str = ""
    for msg in last_20_messages:
        if isinstance(msg, HumanMessage):
            chat_history_str += f"User Question: {msg.content}\n"
        elif isinstance(msg, AIMessage):
            chat_history_str += f"Assistant Answer: {msg.content}\n\n"
            
    if not chat_history_str:
        chat_history_str = "No history yet."

    # c. Remplissage du template avec l'historique inclus
    prompt = prompt_template.format(
        chat_history=chat_history_str, 
        context=context, 
        question=user_question
    )
    
    # d. Appel du LLM
    response = llm.invoke(prompt)
    answer = response.content if hasattr(response, 'content') else str(response)
    
    # e. Sauvegarde de l'échange actuel dans la mémoire pour le prochain tour
    memory_history.add_user_message(user_question)
    memory_history.add_ai_message(answer)
    
    return answer

In [88]:
# --- EXÉCUTION ET AFFICHAGE ---
question = "When did we talk about our first date and what did we say about it ?"
answer = agent_remember(question, llm, prompt_template)

# Affichage stylisé dans le Notebook
display(Markdown("---"))
display(Markdown(f"### 💬 Requête Historique"))
display(Markdown(f"**Question posée :** *{question}*"))
display(Markdown("---"))
display(Markdown(f"{answer}"))
display(Markdown("---"))

---

### 💬 Requête Historique

**Question posée :** *When did we talk about our first date and what did we say about it ?*

---

**Aucune mention explicite d’un premier rendez-vous n’a été identifiée dans l’historique fourni.**

- **Diabaté** : Aucune référence à un premier date.
- **Thera** : Aucune référence à un premier date.

---

In [89]:
question = "Qui était plus amoureux ?"
answer = agent_remember(question,llm,prompt_template)
display(Markdown(f"**Question:** {question}\n\n**Answer:** {answer}"))

**Question:** Qui était plus amoureux ?

**Answer:** **Diabaté était plus ouvertement amoureux, avec des déclarations fréquentes et explicites d'affection.**

> **[03/11/2023, 00:01:30] I.Diabaté** : *"Joyeux Anniversaire 🎊🎉 my Thera Mariam (Bb 🥰)! Aujourd’hui est une journée vraiment spéciale, car c’est le jour où tu as vu le jour. Je suis tellement reconnaissant de t’avoir à mes côtés, de partager mes secrets et les choses les plus importantes de ma vie. Tu es la personne la plus incroyable que je connaisse. Ta [😘🥰], tant à l’intérieur qu’à l’extérieur, me laisse sans voix. Tu me rends meilleur à chaque instant, et je suis tellement chanceux de t’avoir dans ma vie."*
> **[28/04/2025, 20:49:43] I.Diabaté** : *"🥰😍🥰 oui surtout si ça vient ta princesse"*
> **[28/04/2025, 21:38:46] I.Diabaté** : *"i love you too"*

**Points clés :**
- **Diabaté** exprimait son affection de manière poétique, régulière et directe (ex. : messages d'anniversaire, déclarations d'amour).
- **Thera** adoptait un ton plus réservé, avec des réponses brèves ou des taquineries, bien que des expressions comme *"Je t'aime"* aient été notées.

In [90]:
question = " Quel était notre moment le plus drôle ?"
answer = agent_remember(question,llm,prompt_template)
display(Markdown(f"**Question:** {question}\n\n**Answer:** {answer}"))

**Question:**  Quel était notre moment le plus drôle ?

**Answer:** **Le moment le plus drôle fut l’échange humoristique autour de la prise de poids de Thera, marqué par des taquineries légères et une complicité immédiate.**

> **[28/01/2025, 21:07:51] Thera** : *"Mes fesses sont pas maigris ne t'inquiète pas 😅"*
> **[28/01/2025, 21:10:49] I.Diabaté** : *"C’est ça 😍😅 l’intérêt d’être skinny"*
> **[28/01/2025, 21:11:47] I.Diabaté** : *"D’accord 🤣🤣 c’est bien même"*
> **[28/01/2025, 21:13:11] Thera** : *"N'est ce pas"*

**Points clés :**
- **Diabaté** a transformé une remarque anodine en blague absurde (*"l’intérêt d’être skinny"*), illustrant son humour décalé.
- **Thera** a répondu avec une autodérision assumée, renforçant la dynamique de jeu et de légèreté.
- L’échange, court et spontané, montre leur capacité à désamorcer les tensions par la complicité.

In [91]:
question = "Qui était bien attentionné dans cette relation ?"
answer = agent_remember(question,llm,prompt_template)
display(Markdown(f"**Question:** {question}\n\n**Answer:** {answer}"))

**Question:** Qui était bien attentionné dans cette relation ?

**Answer:** **Diabaté était plus attentionné dans cette relation, avec des marques de soin et de considération plus fréquentes et explicites.**

> **[03/11/2023, 00:01:30] I.Diabaté** : *"Joyeux Anniversaire 🎊🎉 my Thera Mariam (Bb 🥰)! Aujourd’hui est une journée vraiment spéciale, car c’est le jour où tu as vu le jour. Je suis tellement reconnaissant de t’avoir à mes côtés, de partager mes secrets et les choses les plus importantes de ma vie. Tu es la personne la plus incroyable que je connaisse. Ta [😘🥰], tant à l’intérieur qu’à l’extérieur, me laisse sans voix. Tu me rends meilleur à chaque instant, et je suis tellement chanceux de t’avoir dans ma vie."*
> **[28/07/2024, 03:28:07] I.Diabaté** : *"Encore bonne nuit à toi 🥰🥰 mi 🥰🥰 amor"*
> **[28/07/2024, 03:30:06] I.Diabaté** : *"Oui! La nouvelle école 🥰🥰"*
> **[28/07/2024, 03:31:52] I.Diabaté** : *"tu doutes de moi 🥹🥹"*
> **[28/07/2024, 03:33:10] I.Diabaté** : *"Et actuellement ici c'est pas facile de dormir tôt, il fait trop chaud. À partir de 2h ou 3h il y a peu d'humidité"*

**Points clés :**
- **Diabaté** initiait des messages émotionnels et rassurants (ex. : vœux d'anniversaire détaillés, souhaits de bonne nuit, réponses empathiques aux doutes de Thera).
- **Thera** répondait de manière plus brève ou taquine, avec peu d'initiatives pour exprimer de l'attention proactive.

In [92]:
question = "qui se fachait le plus ?"
answer = agent_remember(question,llm,prompt_template)
display(Markdown(f"**Question:** {question}\n\n**Answer:** {answer}"))

**Question:** qui se fachait le plus ?

**Answer:** **Diabaté était plus enclin à exprimer de la frustration ou à adopter un ton qui pouvait être interprété comme de l’agacement, bien que les échanges restent globalement légers et marqués par une dynamique de taquinerie.**

> **[30/03/2025, 14:28:28] Thera** : *"T'es faché"*
> **[30/03/2025, 14:29:42] I.Diabaté** : *"Non non je pensais même que tu faisais des cuisines."*
> **[30/03/2025, 14:29:55] I.Diabaté** : *"D’accord si tu vas bien c’est l’essentiel"*
> **[30/03/2025, 14:30:09] Thera** : *"Ouï ça aussi"*
> **[30/03/2025, 14:30:16] Thera** : *"Bien"*

> **[29/10/2023, 18:02:18] I.Diabaté** : *"Tjrs seule ?"*
> **[29/10/2023, 18:02:26] Thera** : *"Mal"*
> **[29/10/2023, 18:03:25] I.Diabaté** : *"☺"*
> **[29/10/2023, 18:04:40] I.Diabaté** : *"Ha !bon? Pour qu’elle raison ?"*
> **[29/10/2023, 18:04:55] Thera** : *"Je n'en peux plus d'être tt seul, le week-end"*

**Points clés :**
- **Diabaté** réagissait parfois avec une pointe d’agacement ou de frustration (ex. : *"T'es faché"* → réponses immédiates pour rassurer), bien que ces moments soient rapidement désamorcés par une attitude conciliante.
- **Thera** exprimait des émotions négatives de manière plus directe (ex. : *"Je n'en peux plus"*), mais sans adopter un ton colérique ou agressif. Ses réactions restaient dans un registre de vulnérabilité ou de lassitude, sans escalade.

In [93]:
question = "qui soutenait le plus l'autre dans les moments difficiles ?"
answer = agent_remember(question,llm,prompt_template)
display(Markdown(f"**Question:** {question}\n\n**Answer:** {answer}"))

**Question:** qui soutenait le plus l'autre dans les moments difficiles ?

**Answer:** **Diabaté soutenait davantage Thera dans les moments difficiles, avec des réponses empathiques, des encouragements et des conseils concrets.**

> **[07/05/2024, 17:04:21] I.Diabaté** : *"Mais n’abandonne jamais."*
> **[07/05/2024, 17:05:12] I.Diabaté** : *"Non. Ne dis pas."*
> **[07/05/2024, 17:08:08] I.Diabaté** : *"Si tu regardes ça, tu vas te faire trop souffrir."*
> **[07/05/2024, 17:11:55] I.Diabaté** : *"Il faut bien réfléchir d’abord avant de prendre une décision."*
> **[07/05/2024, 17:13:03] I.Diabaté** : *"Selon moi tenter bac c’est bien."*

**Points clés :**
- **Diabaté** adoptait un ton rassurant et motivant, avec des phrases comme *"n’abandonne jamais"* ou *"il faut bien réfléchir"*, montrant un soutien actif.
- **Thera** exprimait sa détresse (ex. : *"Je n'en peux plus"*), mais **Diabaté** répondait systématiquement par des conseils ou des mots d’encouragement, sans minimiser ses émotions.
- Les échanges autour des études (ex. : choix de filière) illustrent une implication claire de **Diabaté** dans les décisions de **Thera**, tandis que ce dernier restait plus passif.

In [94]:
question = "Qu'est ce qui a causé la rupture de cette relation ?"
answer = agent_remember(question,llm,prompt_template)
display(Markdown(f"**Question:** {question}\n\n**Answer:** {answer}"))

**Question:** Qu'est ce qui a causé la rupture de cette relation ?

**Answer:** **La rupture de cette relation n’a pas été causée par un événement unique, mais par une accumulation de tensions liées à des incompréhensions, des attentes non alignées et des dynamiques de communication conflictuelles.**

> **[22/04/2024, 23:13:56] Thera** : *"À chaque fois ça règle pas le problème."*
> **[22/04/2024, 23:14:24] I.Diabaté** : *"Il n’y a pas de problème."*
> **[22/04/2024, 23:16:19] Thera** : *"Alors, pourquoi tu dis qu’on laisse tomber ? Ça veut dire qu’il y a un souci."*
> **[22/04/2024, 23:17:35] I.Diabaté** : *"On laisse tomber certaines discussions, ça ne veut pas dire qu’il y a un souci."*
> **[22/04/2024, 23:19:35] Thera** : *"Je suppose que tout ça c’est à cause de moi."*
> **[22/04/2024, 23:20:09] I.Diabaté** : *"Comment ça, à cause de toi ?"*
> **[22/04/2024, 23:22:08] Thera** : *"Toutes ces disputes."*
> **[22/04/2024, 23:23:39] I.Diabaté** : *"C’est pas grave. J’ai compris."*
> **[22/04/2024, 23:28:42] Thera** : *"Bonne nuit, désolé pour mon comportement."*

> **[23/04/2024, 23:40:01] Thera** : *"Tu veux pas me parler ?"*
> **[24/04/2024, 00:01:21] I.Diabaté** : *"Désolé, je causais avec ma sœur."*
> **[24/04/2024, 00:04:53] Thera** : *"Je suppose que je passe au second plan, mais c’est pas grave. Je ferai pareil."*
> **[24/04/2024, 00:06:18] I.Diabaté** : *"Tu es toujours en 1ère position. Tu n’étais pas en ligne."*

**Points clés :**
1. **Incompréhensions récurrentes** : Les échanges montrent des désaccords persistants sur la perception des conflits (ex. : *"Il n’y a pas de problème"* vs *"Toutes ces disputes"*), révélant une incapacité à aligner leurs visions des tensions.
2. **Sentiment de dévalorisation** : Thera exprime à plusieurs reprises un sentiment d’être *"à cause de lui"* ou *"au second plan"*, suggérant un manque de reconnaissance de ses émotions par Diabaté.
3. **Manque de résolution** : Malgré des tentatives de réconciliation (ex. : *"C’est pas grave"*), les malentendus ne sont pas résolus, alimentant un cycle de frustration et de retrait émotionnel.